In [5]:
# This code creates a basic Retrieval-Augmented Generation (RAG) system. It uses LangChain and Google’s Gemini.
# The code does the following:
# 1. Loads Google API key and starts two AI models. One model understands text, and the other generates answers.
# 1a. Initialize Model using LLM gemini-2.5-flash-lite and Embeddings using gemini-embedding-001
# 2. It creates a small library of sentences. The sentences are about RAG and pets.
# 3. It converts the sentences into numbers using FAISS. To find the most relevant sentences quickly.
# 4. When a question is asked, the code searches the library for the two most relevant matches.
# 5. The prompt tells the AI to say it does not know if the answer is not in the notes.
    
import os
from dotenv import load_dotenv
# 1. Fixed Imports
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate  # Corrected from 'ChatPromptTemplates'
from langchain_core.output_parsers import StrOutputParser  # Missing import
from langchain_core.runnables import RunnablePassthrough  # Missing import

def run_rag_query(user_query: str):
    try:
        # 2. Load Environment Variables
        load_dotenv()
        api_key = os.getenv("GOOGLE_API_KEY")
        
        if not api_key:
            raise ValueError("GOOGLE_API_KEY not found. Please check your .env file.")

        # 3. Initialize Model using LLM gemini-2.5-flash-lite and Embeddings using gemini-embedding-001 
        llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", google_api_key=api_key, temperature=0)
        embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
        
        
        # 4. Prepare Knowledge Base
        docs = [
            Document(page_content="RAG stands for Retrieval-Augmented Generation."),
            Document(page_content="RAG helps LLMs produce answers grounded in reality."),
            Document(page_content="The dog is playing."),
            Document(page_content="The cat is running."),
            Document(page_content="the puppy is sleeping.")
        ]

        # 5. Create Vector Store & Retriever
        vectorstore = FAISS.from_documents(docs, embeddings)
        retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

        # 6. Define the Prompt Template
        system_template = (
            "Use the following pieces of retrieved context to answer the question. "
            "If the answer is unknown, state that the answer is unknown. "
            "\n\n"
            "Context: {context}"
        )
        # Roles must be "system" and "human" (not "question")
        prompt = ChatPromptTemplate.from_messages([
            ("system", system_template),
            ("human", "{input}"),
        ])

        # 7. Build the LCEL Chain
        def format_docs(docs):
            return "\n\n".join(doc.page_content for doc in docs)

        rag_chain = (
            {"context": retriever | format_docs, "input": RunnablePassthrough()}


            | prompt
            | llm
            | StrOutputParser()
        )

        # 8. Execute the Query
        print(f"Querying: {user_query}")
        response = rag_chain.invoke(user_query)
        return response

    except Exception as e:
        return f"An error occurred: {str(e)}"

# Example Usage
if __name__ == "__main__":
    result = run_rag_query("What is RAG?")
    print(f"Answer: {result}")
    
    # Test for "unknown" case
    result_unknown = run_rag_query("What is the internet reimbursement policy?")
    print(f"Answer: {result_unknown}")


Querying: What is RAG?
Answer: RAG stands for Retrieval-Augmented Generation. It's a technique that helps Large Language Models (LLMs) produce answers that are based on real-world information.
Querying: What is the internet reimbursement policy?
Answer: The provided context does not contain information about an internet reimbursement policy. Therefore, I cannot answer your question.
